# 1일차 실습 1: HTTP 요청 및 응답 확인

## 실습 목표

이번 실습에서는 Python `requests`를 사용해 HTTP 요청을 보내고 응답을 확인합니다.

단순히 코드를 보는 것이 아니라, 직접 실행하면서 다음을 관찰합니다.

- GET 요청과 POST 요청의 차이
- Status Code의 의미
- Header와 Body의 역할
- JSON 응답을 Python 객체로 다루는 방법
- Query Parameter 전달 방식
- JSON Body 전송 방식
- LLM 서비스에서 사용할 수 있는 요청 Body 형태

> 이 실습에서는 JSONPlaceholder 테스트 API를 사용합니다. 실제 데이터가 저장되지는 않습니다.


## 1. GET 요청 보내기

GET 요청은 서버에서 데이터를 조회할 때 주로 사용합니다.

아래 코드는 테스트 API에서 게시글 1개를 조회합니다.


In [1]:
import requests
import json
from pprint import pprint

url = "https://jsonplaceholder.typicode.com/posts/1"

response = requests.get(url)

print(response)

<Response [200]>


`response` 객체 자체를 출력하면 응답의 요약만 보입니다.  
실제 응답 내용을 확인하려면 Status Code, Header, Body를 따로 살펴봐야 합니다.


In [2]:
print("Status Code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Body:")
print(response.text)

Status Code: 200
Content-Type: application/json; charset=utf-8
Body:
{
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
  "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
}


## 2. Status Code 확인하기

Status Code는 요청 처리 결과를 숫자로 알려줍니다.

| Status Code | 의미 |
|---:|---|
| 200| 요청 성공 |
| 201 | 생성 성공 |
| 400 | 잘못된 요청 |
| 404 | 요청한 주소를 찾을 수 없음 |
| 500 | 서버 내부 오류 |

아래 코드에서 응답이 성공인지 확인해봅니다.


In [3]:
if response.status_code == 200:
    print("요청이 성공했습니다.")
else:
    print("요청이 실패했습니다.")

요청이 성공했습니다.


## 3. Header 확인하기

Header는 요청이나 응답에 대한 부가 정보를 담습니다.

예를 들어 `Content-Type`을 보면 응답 Body가 어떤 형식인지 알 수 있습니다.


In [4]:
print("Content-Type:", response.headers.get("Content-Type"))

Content-Type: application/json; charset=utf-8


전체 Header도 확인해봅니다.  
실제 API를 다룰 때는 인증 정보, 데이터 형식, 캐시 정책 등도 Header에 포함될 수 있습니다.


In [5]:
for key, value in response.headers.items():
    print(f"{key}: {value}")

Date: Wed, 10 Jun 2026 02:50:59 GMT
Content-Type: application/json; charset=utf-8
Transfer-Encoding: chunked
Connection: keep-alive
Access-Control-Allow-Credentials: true
Cache-Control: max-age=43200
Etag: W/"124-yiKdLzqO5gfBrJFrcdJ8Yq0LGnU"
Expires: -1
Nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}
Pragma: no-cache
Report-To: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=Y87BfwfGkmjrlwqs4hyPA%2Bz1%2BSPr5zLA8I2aENVWM4g%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1754470020"}],"max_age":3600}
Reporting-Endpoints: heroku-nel="https://nel.heroku.com/reports?s=Y87BfwfGkmjrlwqs4hyPA%2Bz1%2BSPr5zLA8I2aENVWM4g%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1754470020"
Server: cloudflare
Vary: Origin, Accept-Encoding
Via: 2.0 heroku-router
X-Content-Type-Options: nosniff
X-Powered-By: Express
X-Ratelimit-Limit: 1000
X-Ratelimit-Remaining: 999
X-Ratelimit-Reset: 1754470024
Age: 3

## 4. JSON 응답을 Python 객체로 변환하기

응답 Body가 JSON 형식이면 `response.json()`을 사용해 Python dict로 바꿀 수 있습니다.


In [6]:
data = response.json()

print(type(data))
pprint(data)

<class 'dict'>
{'body': 'quia et suscipit\n'
         'suscipit recusandae consequuntur expedita et cum\n'
         'reprehenderit molestiae ut ut quas totam\n'
         'nostrum rerum est autem sunt rem eveniet architecto',
 'id': 1,
 'title': 'sunt aut facere repellat provident occaecati excepturi optio '
          'reprehenderit',
 'userId': 1}


dict가 되었으므로 key를 사용해 필요한 값만 꺼낼 수 있습니다.


In [7]:
print("게시글 ID:", data["id"])
print("제목:", data["title"])
print("본문:", data["body"])

게시글 ID: 1
제목: sunt aut facere repellat provident occaecati excepturi optio reprehenderit
본문: quia et suscipit
suscipit recusandae consequuntur expedita et cum
reprehenderit molestiae ut ut quas totam
nostrum rerum est autem sunt rem eveniet architecto


## 5. 응답 요약 함수 만들기

여러 API 요청을 반복해서 테스트할 것이므로, 응답을 보기 좋게 출력하는 함수를 만들어둡니다.


In [8]:
def print_response_summary(response):
    print("=" * 60)
    print("Status Code:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))
    print("-" * 60)

    try:
        body = response.json()
        print("JSON Body:")
        print(json.dumps(body, ensure_ascii=False, indent=2))
    except Exception:
        print("Text Body:")
        print(response.text)

    print("=" * 60)


print_response_summary(response)

Status Code: 200
Content-Type: application/json; charset=utf-8
------------------------------------------------------------
JSON Body:
{
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
  "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
}


## 6. Query Parameter 사용하기

Query Parameter는 URL 뒤에 붙는 입력값입니다.

예를 들어 특정 사용자의 게시글만 조회하고 싶을 때 다음과 같은 형태를 사용할 수 있습니다.

```text
/posts?userId=1
```

Python `requests`에서는 `params` 인자로 Query Parameter를 전달합니다.


In [21]:
url = "https://jsonplaceholder.typicode.com/posts"

params = {
    "userId": 10
}

response = requests.get(url, params=params)

print_response_summary(response)

Status Code: 200
Content-Type: application/json; charset=utf-8
------------------------------------------------------------
JSON Body:
[
  {
    "userId": 10,
    "id": 91,
    "title": "aut amet sed",
    "body": "libero voluptate eveniet aperiam sed\nsunt placeat suscipit molestias\nsimilique fugit nam natus\nexpedita consequatur consequatur dolores quia eos et placeat"
  },
  {
    "userId": 10,
    "id": 92,
    "title": "ratione ex tenetur perferendis",
    "body": "aut et excepturi dicta laudantium sint rerum nihil\nlaudantium et at\na neque minima officia et similique libero et\ncommodi voluptate qui"
  },
  {
    "userId": 10,
    "id": 93,
    "title": "beatae soluta recusandae",
    "body": "dolorem quibusdam ducimus consequuntur dicta aut quo laboriosam\nvoluptatem quis enim recusandae ut sed sunt\nnostrum est odit totam\nsit error sed sunt eveniet provident qui nulla"
  },
  {
    "userId": 10,
    "id": 94,
    "title": "qui qui voluptates illo iste minima",
    "body": "a

응답이 게시글 1개가 아니라 목록입니다.  
JSON 배열은 Python에서는 list로 변환됩니다.


In [22]:
posts = response.json()

print(type(posts))
print("게시글 수:", len(posts))
print("첫 번째 게시글:")
pprint(posts[0])

<class 'list'>
게시글 수: 10
첫 번째 게시글:
{'body': 'libero voluptate eveniet aperiam sed\n'
         'sunt placeat suscipit molestias\n'
         'similique fugit nam natus\n'
         'expedita consequatur consequatur dolores quia eos et placeat',
 'id': 91,
 'title': 'aut amet sed',
 'userId': 10}


Query Parameter 값을 바꿔보면서 결과가 어떻게 달라지는지 확인합니다.


In [11]:
for user_id in [1, 2, 3]:
    params = {"userId": user_id}
    response = requests.get(url, params=params)
    posts = response.json()

    print(f"userId={user_id} 게시글 수:", len(posts))
    print("첫 번째 제목:", posts[0]["title"])
    print("-" * 60)

userId=1 게시글 수: 10
첫 번째 제목: sunt aut facere repellat provident occaecati excepturi optio reprehenderit
------------------------------------------------------------
userId=2 게시글 수: 10
첫 번째 제목: et ea vero quia laudantium autem
------------------------------------------------------------
userId=3 게시글 수: 10
첫 번째 제목: asperiores ea ipsam voluptatibus modi minima quia sint
------------------------------------------------------------


## 7. POST 요청 보내기

POST 요청은 서버에 데이터를 보낼 때 주로 사용합니다.

사용자 입력, 게시글 작성 내용, 챗봇 질문 등은 보통 요청 Body에 담아 서버로 전달합니다.


In [12]:
url = "https://jsonplaceholder.typicode.com/posts"

payload = {
    "title": "hello",
    "body": "API practice",
    "userId": 1
}

response = requests.post(url, json=payload)

print_response_summary(response)

Status Code: 201
Content-Type: application/json; charset=utf-8
------------------------------------------------------------
JSON Body:
{
  "title": "hello",
  "body": "API practice",
  "userId": 1,
  "id": 101
}


`json=payload`로 전달하면 Python dict가 JSON Body로 전송됩니다.  
우리가 서버로 보낸 데이터를 다시 확인해봅니다.


In [13]:
print("요청 Body로 보낸 데이터:")
print(json.dumps(payload, ensure_ascii=False, indent=2))

요청 Body로 보낸 데이터:
{
  "title": "hello",
  "body": "API practice",
  "userId": 1
}


## 8. Header를 명시해서 POST 요청 보내기

`json=payload`를 사용하면 `Content-Type: application/json` Header가 자동으로 설정됩니다.

이번에는 Header를 직접 지정해봅니다.


In [14]:
headers = {
    "Content-Type": "application/json"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)

print_response_summary(response)

Status Code: 201
Content-Type: application/json; charset=utf-8
------------------------------------------------------------
JSON Body:
{
  "title": "hello",
  "body": "API practice",
  "userId": 1,
  "id": 101
}


## 9. 실패 응답 확인하기

존재하지 않는 주소로 요청을 보내면 실패 응답이 돌아옵니다.

실패 응답도 Status Code와 Body를 확인해야 원인을 추정할 수 있습니다.


In [15]:
wrong_url = "https://jsonplaceholder.typicode.com/not-found-url"

response = requests.get(wrong_url)

print_response_summary(response)

Status Code: 404
Content-Type: application/json; charset=utf-8
------------------------------------------------------------
JSON Body:
{}


## 10. LLM 서비스 요청 Body 형태 살펴보기

아직 실제 LLM API를 호출하지는 않습니다.

다만 LLM 챗 API에 사용자 질문을 보낸다면 요청 Body는 아래처럼 구성될 수 있습니다.


In [16]:
chat_payload = {
    "message": "FastAPI가 뭐야?",
    "temperature": 0.7
}

print(json.dumps(chat_payload, ensure_ascii=False, indent=2))

{
  "message": "FastAPI가 뭐야?",
  "temperature": 0.7
}


요청 Body는 결국 서버에 전달할 입력 데이터입니다.  
LLM 서비스에서는 `message`에 사용자의 질문이 들어가고, 옵션 값이 함께 전달될 수 있습니다.


# TODO 실습

지금까지는 코드를 실행하며 HTTP 요청과 응답을 관찰했습니다.

이제부터는 직접 작은 함수를 구현해봅니다.  
TODO는 실력 확인용이며, 앞에서 실습한 내용을 함수로 정리하는 단계입니다.


## TODO 1. 게시글 하나를 조회하는 함수 만들기

### 목표

게시글 ID를 입력받아 해당 게시글을 조회하는 `get_post(post_id)` 함수를 완성하세요.

### 요구사항

- `https://jsonplaceholder.typicode.com/posts/{post_id}`로 GET 요청을 보냅니다.
- 요청이 성공하면 JSON 데이터를 반환합니다.
- 요청이 실패하면 `None`을 반환합니다.

### 입력 예시

```python
get_post(2)
```


In [17]:
def get_post(post_id):
    base_url = "https://jsonplaceholder.typicode.com/posts"
    url = f"{base_url}/{post_id}"

    response = requests.get(url)

    # TODO 1:
    # 응답이 성공이면 response.json()을, 실패라면 None을 반환하세요.

    pass


# 테스트
for post_id in [2, 999999]:
    print("=" * 60)
    print(f"조회할 게시글 ID: {post_id}")

    post = get_post(post_id)

    if post is None:
        print("조회 결과: 게시글을 찾을 수 없습니다.")
    else:
        print("조회 결과: 성공")
        print("게시글 ID:", post["id"])
        print("제목:", post["title"])
        print("본문 미리보기:", post["body"][:60] + "...")


조회할 게시글 ID: 2
조회 결과: 게시글을 찾을 수 없습니다.
조회할 게시글 ID: 999999
조회 결과: 게시글을 찾을 수 없습니다.


## TODO 2. 조회 결과에서 필요한 정보만 추출하기

### 목표

TODO 1에서 만든 `get_post(post_id)`를 활용해, 게시글의 핵심 정보만 정리하는 `get_post_summary(post_id)` 함수를 완성하세요.

### 요구사항

- 내부에서 `get_post(post_id)`를 호출합니다.
- 게시글이 존재하지 않으면 `None`을 반환합니다.
- 게시글이 존재하면 아래 형태의 dict를 반환합니다.

```python
{
    "id": 2,
    "title": "...",
    "body_preview": "본문 앞 40자...",
    "body_length": 120
}
```

### 확인할 점

이 TODO는 HTTP 요청 자체보다, API 응답 JSON에서 필요한 필드를 꺼내고 가공하는 연습입니다.


In [ ]:
def get_post_summary(post_id):
    post = get_post(post_id)

    # TODO 2-1:
    # post가 None이면 None을 반환하세요.

    # TODO 2-2:
    # post에서 id, title, body 값을 꺼내세요.

    # TODO 2-3:
    # body_preview는 body의 앞 40자만 사용하세요.
    # body_length는 body 전체 길이로 계산하세요.

    # TODO 2-4:
    # 요구사항에 맞는 dict를 반환하세요.
    
    pass


# 테스트
for post_id in [2, 999999]:
    print("=" * 60)
    print(f"요약할 게시글 ID: {post_id}")

    summary = get_post_summary(post_id)

    if summary is None:
        print("요약 결과: 게시글을 찾을 수 없습니다.")
    else:
        print("요약 결과: 성공")
        print("게시글 ID:", summary["id"])
        print("제목:", summary["title"])
        print("본문 미리보기:", summary["body_preview"])
        print("본문 길이:", summary["body_length"])


## TODO 3. POST 요청 함수 만들기

### 목표

제목, 본문, 사용자 ID를 입력받아 POST 요청을 보내는 `create_post(title, body, user_id)` 함수를 완성하세요.

### 요구사항

- 입력값으로 `payload` dict를 구성합니다.
- `requests.post(..., json=payload)`로 요청을 보냅니다.
- 응답 JSON을 반환합니다.


In [ ]:
def create_post(title, body, user_id):
    url = "https://jsonplaceholder.typicode.com/posts"

    # TODO 3-1:
    # title, body, userId를 포함한 payload를 만드세요.

    # TODO 3-2:
    # POST 요청을 보내세요.

    # TODO 3-3:
    # 응답 JSON을 반환하세요.
    
    pass


# 테스트
created = create_post(
    title="my first api request",
    body="I am learning backend api",
    user_id=10
)

print("=" * 60)
print("POST 요청 결과")

if created is None:
    print("생성 결과: 응답이 없습니다. TODO 코드를 확인하세요.")
else:
    print("생성 결과: 성공")
    print("생성된 ID:", created.get("id"))
    print("제목:", created.get("title"))
    print("본문:", created.get("body"))
    print("사용자 ID:", created.get("userId"))


## TODO 4. LLM 챗 요청 Body 생성 함수 만들기

### 목표

사용자 질문을 입력받아 LLM 챗 API에 보낼 수 있는 요청 Body를 만드는 `build_chat_payload(message, temperature=0.7)` 함수를 완성하세요.

### 요구사항

- `message`가 비어 있으면 `ValueError`를 발생시킵니다.
- 정상 입력이면 `message`, `temperature`를 담은 dict를 반환합니다.


In [ ]:
def build_chat_payload(message, temperature=0.7):
    # TODO 4-1:
    # message가 비어 있으면 ValueError를 발생시키세요.

    # TODO 4-2:
    # 정상 입력이면 message와 temperature를 담은 dict를 반환하세요.
    pass


# 테스트 1: 정상 입력
print("=" * 60)
print("테스트 1: 정상 입력")

payload = build_chat_payload("오늘 배운 HTTP 요청 구조를 요약해줘")
print("생성된 요청 Body:")
print(json.dumps(payload, ensure_ascii=False, indent=2))

# 테스트 2: 빈 입력
print("=" * 60)
print("테스트 2: 빈 입력")

try:
    build_chat_payload("")
except ValueError as e:
    print("예외 발생:", e)


## 실습 정리

이번 실습에서는 HTTP 요청과 응답을 직접 확인하고, 함수로 정리해보았습니다.

정리하면 다음과 같습니다.

- GET 요청은 데이터를 조회할 때 사용합니다.
- POST 요청은 서버에 데이터를 보낼 때 사용합니다.
- Query Parameter는 조회 조건을 전달할 때 사용할 수 있습니다.
- JSON Body는 서버에 전달할 실제 입력 데이터를 담습니다.
- LLM 서비스에서도 사용자의 질문은 JSON Body 형태로 서버에 전달될 수 있습니다.
